In [0]:
import json
import re
from datetime import date


dbutils.widgets.dropdown("environment", "dev", ["dev"])
dbutils.widgets.text("requested_run_date", "")
dbutils.widgets.text("timeout_seconds", "7200")
dbutils.widgets.text("max_llm_calls", "1000")
environment = dbutils.widgets.get("environment")
requested_run_date = dbutils.widgets.get("requested_run_date").strip()
if requested_run_date:
    date.fromisoformat(requested_run_date)
if environment != "dev":
    raise ValueError(f"Unsupported environment: {environment}")
timeout_text = dbutils.widgets.get("timeout_seconds")
if not re.fullmatch(r"\d+", timeout_text) or not (int(timeout_text) == 0 or 60 <= int(timeout_text) <= 14400):
    raise ValueError("timeout_seconds must be 0 (no timeout) or between 60 and 14400")
timeout_seconds = int(timeout_text)
limit_text = dbutils.widgets.get("max_llm_calls")
if limit_text != "all" and (not re.fullmatch(r"\d+", limit_text) or not 1 <= int(limit_text) <= 1000):
    raise ValueError("max_llm_calls must be 'all' or between 1 and 1000")

steps = []
try:
    refresh = json.loads(dbutils.notebook.run("./EIC OG Refresh", timeout_seconds, {
        "environment": environment, "requested_run_date": requested_run_date,
        "max_llm_calls": limit_text,
    }))
    if not re.fullmatch(r"\d{8}T\d{6}Z", refresh.get("run_utc", "")):
        raise ValueError("Refresh did not return a valid run_utc")
    steps.append({"step": "refresh", "status": "published", "run_utc": refresh["run_utc"]})
    validation = json.loads(dbutils.notebook.run("./EIC OG Validate", timeout_seconds, {
        "run_utc": refresh["run_utc"],
        "expected_pairs": str(refresh["candidate_pairs"]),
        "expected_pending": str(refresh["pending_pairs"]),
        "expected_without_candidates": str(refresh["projects_without_candidates"]),
        "expected_possible_pairs": str(refresh["possible_pairs"]),
        "expected_screened_pairs": str(refresh["screened_pairs"]),
    }))
    if validation.get("status") != "validated" or validation.get("run_utc") != refresh["run_utc"]:
        raise ValueError("Validation result does not match the published run")
    steps.append({"step": "validation", "status": "validated", "run_utc": refresh["run_utc"]})
    if limit_text == "all" and (refresh.get("pending_pairs", 0) or refresh.get("llm_errors", 0)):
        raise ValueError(f"Unfinished model review: pending_pairs={refresh['pending_pairs']}, "
                         f"llm_errors={refresh.get('llm_errors', 0)}")
except Exception as exc:
    print(json.dumps({"status": "failed", "steps": steps, "error": str(exc)}, sort_keys=True))
    raise

result = {"status": "batch_published", "steps": steps, "summary": refresh,
          "pending_pairs": refresh["pending_pairs"], "validation": validation}
print(json.dumps(result, sort_keys=True))
dbutils.notebook.exit(json.dumps(result))